In [ ]:
import joblib
file_path = "/root/autodl-tmp/DATA/data_bundles.pkl"
data_bundles = joblib.load(file_path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils import resample
import os

# 输出目录
out_dir = "/root/autodl-tmp/Figure2"
os.makedirs(out_dir, exist_ok=True)

ratio = 1
n_runs = 1
random_state = 42
perplexity_values = [5]
variance_threshold = 0.90

for set_name in ["set1", "set2", "set3"]:

    print(f"\n========== Processing {set_name} ==========\n")

    # === 合并 train + test ===
    df_train = data_bundles[set_name]["train"]
    df_test  = data_bundles[set_name]["test"]
    df = pd.concat([df_train, df_test], axis=0)

    # === 分离 X 和 y ===
    y = df["insider_trading"]
    id_cols = ["Stkcd", "Trddt"]
    X = df.drop(columns=["insider_trading"] + id_cols, errors="ignore")

    df_pos, y_pos = X[y == 1], y[y == 1]
    df_neg, y_neg = X[y == 0], y[y == 0]

    print("Positive:", len(df_pos))
    print("Negative:", len(df_neg))

    for run in range(1, n_runs + 1):

        # === 1:1 抽样 ===
        n_pos = len(df_pos)
        n_neg = n_pos * ratio

        df_neg_resampled, y_neg_resampled = resample(
            df_neg,
            y_neg,
            n_samples=n_neg,
            random_state=random_state + run
        )

        df_sampled = pd.concat([df_pos, df_neg_resampled], axis=0)
        y_sampled  = pd.concat([y_pos, y_neg_resampled], axis=0)

        print(f"Run {run} class distribution:")
        print(y_sampled.value_counts())

        # === 清洗 + 标准化 ===
        df_sampled = df_sampled.replace([np.inf, -np.inf], np.nan).fillna(0)

        scaler = StandardScaler()
        df_scaled = scaler.fit_transform(df_sampled)

        # === PCA 保留90%方差 ===
        pca = PCA(n_components=variance_threshold, random_state=random_state)
        pca_result = pca.fit_transform(df_scaled)

        print(f"PCA components selected: {pca.n_components_}")

        # === t-SNE ===
        for perp in perplexity_values:

            tsne = TSNE(
                n_components=2,
                perplexity=perp,
                learning_rate="auto",
                n_iter=2000,
                init="pca",
                random_state=random_state + run
            )

            tsne_result = tsne.fit_transform(pca_result)

            # === 可视化 ===
            colors = np.where(y_sampled == 1, 'red', 'black')

            plt.figure(figsize=(6, 6))
            plt.gca().set_aspect('equal', adjustable='box')
            
            plt.scatter(
                tsne_result[:, 0],
                tsne_result[:, 1],
                c=colors,
                alpha=0.5,
                s=5
            )
            
            legend_handles = [
                plt.Line2D([0], [0], marker='o', color='w',
                           label='Non-Insider Trading (0)',
                           markerfacecolor='black', markersize=6),
                plt.Line2D([0], [0], marker='o', color='w',
                           label='Insider Trading (1)',
                           markerfacecolor='red', markersize=6)
            ]
            
            plt.legend(handles=legend_handles,
                       loc='upper center',
                       bbox_to_anchor=(0.5, -0.1),
                       ncol=2,
                       fontsize=10,
                       frameon=False)
            
            # ========== 关键修改：统一坐标轴范围 ==========
            # 强制设置x轴和y轴的范围，确保所有图片方向一致且刻度一致
            plt.xlim(-200, 200)
            plt.ylim(-200, 200)

            
            # 可选：显式设置刻度，确保200一定出现
            plt.xticks([-200, -150, -100, -50, 0, 50, 100, 150, 200])
            plt.yticks([-200, -150, -100, -50, 0, 50, 100, 150, 200])
            
            plt.xlabel('Component 1')
            plt.ylabel('Component 2')
            
            out_path = os.path.join(
                out_dir,
                f"{set_name}_PCA_tSNE_1to1_perp{perp}.png"
            )
            
            plt.savefig(out_path, dpi=300,
                        bbox_inches="tight",
                        transparent=True)

            print(f"Saved: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os

# ============ 配置项（根据你的实际路径修改） ============
# 图片所在目录
out_dir = "/root/autodl-tmp/Figure2"
# 要拼接的三张图片文件名
image_files = [
    "set1_PCA_tSNE_1to1_perp5.png",
    "set2_PCA_tSNE_1to1_perp5.png",
    "set3_PCA_tSNE_1to1_perp5.png"
]
# 对应子图标题
titles = [
    "Y1_PCA +t-SNE Visualization",
    "Y2_PCA +t-SNE Visualization",
    "Y3_PCA +t-SNE Visualization"
]
# 输出文件名
output_file = "combined_horizontal_transparent.png"

# ============ 核心拼接逻辑 ============
# 1. 读取所有图片并验证
imgs = []
for idx, f in enumerate(image_files):
    img_path = os.path.join(out_dir, f)
    if not os.path.exists(img_path):
        raise FileNotFoundError(f"图片文件不存在：{img_path}")
    # 读取图片（保留透明通道）
    img = mpimg.imread(img_path)
    imgs.append(img)
    print(f"成功读取图片 {idx+1}: {f} (尺寸: {img.shape})")

# 2. 创建横向拼接的画布（matplotlib实现，更稳定）
fig, axes = plt.subplots(1, 3, figsize=(24, 8), dpi=300)  # 1行3列，总尺寸24x8英寸
fig.patch.set_alpha(0.0)  # 设置整个画布背景透明

# 3. 逐个绘制子图
for idx, (ax, img, title) in enumerate(zip(axes, imgs, titles)):
    # 绘制图片（保留透明）
    ax.imshow(img)
    # 添加标题
    ax.set_title(title, fontsize=16, color='black', pad=10)  # 白色标题，避免遮挡
    # 隐藏坐标轴（和原图一致）black
    ax.axis('off')
    # 设置子图背景透明
    ax.patch.set_alpha(0.0)

# 4. 调整子图间距，避免重叠
plt.tight_layout()

# 5. 保存图片（关键：设置transparent=True保留透明）
save_path = os.path.join(out_dir, output_file)
plt.savefig(
    save_path,
    dpi=600,
    bbox_inches='tight',
    transparent=True,  # 核心：透明背景
    pad_inches=0.1     # 少量边距，避免裁剪
)
plt.close()

print(f"透明背景的横向图片已保存至：{save_path}")